# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_Session_5_A2A_MCP_Governed.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>


# About This Notebook: Sandboxing Tool Execution

You will build a governed **Agent-to-Agent (A2A)** layer that sits between the LLM agent and **Model Context Protocol (MCP)** servers. In this architecture, the agent never calls MCP directly; it is effectively "air-gapped" from the external world. Instead, it emits structured **ToolRequest** objects—declarations of intent that must pass through a sovereign enforcement layer.

The A2A governor acts as a deterministic firewall. It enforces a rigorous multi-stage policy including explicit allowlists, resource budgets, logical separation rules, and strict argument gates. When the governor encounters a restricted action, it pauses execution via **LangGraph interrupts**, logging the state and waiting for an external approval signal to resume. Every decision—whether a grant, a denial, or a pause—is streamed as standardized A2A events and captured as machine-readable JSON artifacts.

**Important limitation (notebook demo)**: This notebook optionally uses console input to simulate human-in-the-loop (HITL) approval. In a production environment, this synchronous block would be replaced by an asynchronous, out-of-band approval system (e.g., Slack, a ticketing system, or a dedicated Governance UI) that sends a `resume` command back to the graph state.

### Production Gotchas & Architectural Nuances

* **Tool categories:** Move beyond simple string matching. In production, use a registry populated dynamically from tool discovery, ensuring that "Execution" tools cannot masquerade as "Discovery" tools through name heuristics.
* **Approval flow:** Ensure interrupts are truly persistent. Production-grade systems must resume from an external signal (webhook/UI) to handle long-running human latency without timing out.
* **Separation scope:** Mutual exclusion between connection and execution steps is enforced per run here. For multi-turn tasks, you must persist these "contamination flags" in your state store to maintain the risk boundary.
* **LLM proposals:** Governance starts before the request. Validate proposed tool names against the policy at the reasoning stage to prevent the agent from even attempting to formulate a forbidden request.
* **Capability quotas:** Implement quotas at the category level. This prevents "Alias Evasion," where an agent might bypass a per-tool limit by rotating between several tools that achieve the same high-risk outcome.
* **Artifact filtering:** Centralize artifact generation. Enforce allowed artifact types at a single wrapper to prevent the "silent leak" of internal reasoning or sensitive metadata to the logs.
* **Session lifecycle:** Manage MCP sessions within the scope of the individual task. Connecting and disconnecting in the same task lifecycle avoids orphaned sockets and async cancel-scope issues.

### The Six Enforcement Mechanisms

1. **Tool Allowlist (The Gate):** Explicit, pattern-based policy ensuring that only vetted, reachable tools are exposed to the agentic reasoning layer.
2. **Budget Limits (The Wallet):** Hard constraints on total tool calls per task, maximum parallel execution, and total wall-clock runtime to prevent runaway loops and cost spikes.
3. **Argument Gates (The Inspector):** Validation logic that inspects tool arguments *before* execution, catching "broad actions" (e.g., deleting `*`) or excessively large search queries.
4. **Connection Separation (The Protocol):** A stateful rule enforcing mutual exclusion; an agent cannot perform discovery/connection and privileged execution within the same execution turn.
5. **Approval Workflows (The Judge):** Native Human-in-the-loop (HITL) triggers for tools marked as "Restricted," forcing a state-save and interrupt until a human authorizes the action.
6. **Structured Artifacts (The Scribe):** Automatic generation of machine-readable decision logs for every tool call, providing an immutable audit trail for compliance and post-incident analysis.


### Config-as-policy (OmegaConf) and PRP-style skills

* **YAML + OmegaConf:** The same `governance/` and `skills/` trees ship in-repo as `demos/config/`, and are also published [on Google Drive](https://drive.google.com/drive/folders/17gH3w247NtejX4911wFGXQ_8dPUVlHhe?usp=sharing) so you can download the folder once and use `./config/` next to your working directory (no per-session upload). Profile YAML merges on top (e.g. `defaults.yaml` + `discovery_only.yaml`). Use `${oc.env:VAR,default}` so ops can tune budgets without code changes.
* **Config resolution:** `GOVERNANCE_CONFIG_DIR` → `./config` (Drive/local folder) → bundled `demos/config`.

**Why OmegaConf?** You *could* load YAML with PyYAML alone; OmegaConf earns its place when policy is **layered** and **environment-specific**:
* **Structured merge:** `OmegaConf.merge(base, profile)` composes shared defaults with a profile (e.g. discovery vs restricted) without hand-written deep-merge logic or giant duplicated Python literals.
* **Overrides without editing files:** Built-in interpolation (e.g. `${oc.env:DISCOVERY_MAX_RUNTIME,120}`) lets you tune limits per deploy or Colab run via environment variables.
* **Easier maintenance and review:** Policy stays small text files with clear diffs in git or Drive updates; security or ops stakeholders can reason about YAML without touching governor code.
* **Clean handoff to Python:** `OmegaConf.to_container(..., resolve=True)` yields a plain dict tree that maps directly into your existing `GovernancePolicy` dataclass—so the enforcement layer stays the same while config moves out of the code.

* **PRP / agent skill bundle:** Under `skills/` inside that config root, a small YAML file carries *human* intent (`objective`, `constraints`) plus *machine* rules such as `forbidden_argument_substrings`. Skills **narrow** what the base policy already permits—they cannot whitelist tools the policy forbids.
* **Pydantic:** `ToolRequestModel` validates JSON tool proposals from the LLM before they become `ToolRequest` dataclasses.

**NOTE:** This notebook demonstrates governance at the execution boundary, not cryptographic trust between services; identity, attestation, and remote isolation are complementary concerns.


## Installation

In [ ]:
%pip install -q "google-adk[a2a]" "a2a-sdk>=0.3.0" langgraph langchain httpx nest-asyncio uvicorn "mcp>=1.11.0" python-dotenv langchain_openai exa_py omegaconf pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00


## API Keys Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
EXA_API_KEY = os.getenv('EXA_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

if not OPENROUTER_API_KEY:
    print("⚠️  OPENROUTER_API_KEY not set")
if not EXA_API_KEY:
    print("⚠️  EXA_API_KEY not set")
if not SERPAPI_API_KEY:
    print("⚠️  SERPAPI_API_KEY not set")

## Core Imports

In [ ]:
import asyncio
import threading
import time
import contextlib
import httpx
import uvicorn
import nest_asyncio
import json
from pathlib import Path

from omegaconf import OmegaConf
from pydantic import BaseModel, Field
import re
from typing import TypedDict, List, Dict, Any, Optional, Set, Literal
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from contextlib import AsyncExitStack

# A2A imports
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import (
    AgentCapabilities, AgentCard, AgentSkill, TransportProtocol,
    Part, TextPart, TaskState
)
from a2a.utils import new_agent_text_message, new_task
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH

# MCP imports
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Exa imports
from exa_py import Exa

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict as LangGraphTypedDict

# Apply nest_asyncio for Jupyter compatibility
nest_asyncio.apply()

print("✅ Imports loaded")


✅ Imports loaded


## 1. Tool Request and Result Schemas

Define typed schemas for LangGraph to produce requests and receive results.

In [ ]:
@dataclass
class ToolRequest:
    """Structured tool request from LangGraph agent."""
    tool_name: str
    arguments: Dict[str, Any]
    rationale: str = ""
    expects_artifact_names: List[str] = field(default_factory=list)
    stop_after: bool = False


@dataclass
class ToolResult:
    """Structured result from governed execution."""
    tool_name: str
    allowed: bool
    decision_reason: str
    output_summary: str = ""
    raw_output_ref: Optional[str] = None
    policy_match: Optional[str] = None
    policy_pattern: Optional[str] = None
    extracted_tool_slugs: List[str] = field(default_factory=list)
    budget_stats_snapshot: Dict[str, Any] = field(default_factory=dict)

    def to_json(self) -> str:
        """Serialize to JSON for artifacts."""
        return json.dumps({
            "tool_name": self.tool_name,
            "allowed": self.allowed,
            "decision_reason": self.decision_reason,
            "output_summary": self.output_summary[:500],
            "policy_match": self.policy_match,
            "policy_pattern": self.policy_pattern,
            "extracted_tool_slugs": self.extracted_tool_slugs,
            "budget_stats": self.budget_stats_snapshot
        }, indent=2)


@dataclass
class FinalAnswer:
    """Terminal response from agent."""
    content: str
    tools_used: int = 0




class ToolRequestModel(BaseModel):
    """Pydantic validation for LLM-emitted tool JSON before dataclass ToolRequest."""

    tool_name: str
    arguments: Dict[str, Any] = Field(default_factory=dict)
    rationale: str = ""
    expects_artifact_names: List[str] = Field(default_factory=list)
    stop_after: bool = False

    def to_tool_request(self) -> ToolRequest:
        return ToolRequest(
            tool_name=self.tool_name,
            arguments=dict(self.arguments),
            rationale=self.rationale,
            expects_artifact_names=list(self.expects_artifact_names),
            stop_after=self.stop_after,
        )


print("✅ Tool request and result schemas defined")


✅ Tool request and result schemas defined


## 2. Policy Definition with Tool Categorization

Define governance policies and tool categories for connection separation.

## 2a. OmegaConf policy packs and PRP-style skills

**Policy packs:** `governance/defaults.yaml` (under your config root) holds shared limits and artifact types. `discovery_only.yaml` overrides name, budgets, and `tool_policies`. At runtime, `OmegaConf.merge(base, profile)` composes them; `OmegaConf.to_container(..., resolve=True)` materializes env interpolation.

**Why OmegaConf here (recap):** (1) **merge** = shared base + profile without custom merge code; (2) **interpolation** = env-driven overrides; (3) **maintainability** = small YAML files and readable diffs; (4) **resolve → dict** = feed the same `GovernancePolicy` builder. Plain YAML loaders do not give you merge + `${oc.env:...}` in one place.

**PRP-style skill:** `skills/research_strict.yaml` is analogous to a Cursor skill: prose for reviewers, plus `forbidden_argument_substrings` enforced in `governed_call` *after* base policy and argument gates—an extra shrink-wrap on allowed work.

**Working directory:** Run **§2b** below once if you use Colab or a checkout without `demos/config/`. Otherwise the notebook uses bundled `demos/config`, or `./config` after you download the [Drive folder](https://drive.google.com/drive/folders/17gH3w247NtejX4911wFGXQ_8dPUVlHhe?usp=sharing).


## 2b. Shared config folder (Google Drive)

The course keeps an up-to-date copy of the `governance/` and `skills/` trees [on Google Drive](https://drive.google.com/drive/folders/17gH3w247NtejX4911wFGXQ_8dPUVlHhe?usp=sharing). **Download the whole folder** (Drive → Download) and unzip it so your machine has:

`config/governance/defaults.yaml` and `config/skills/...`

with your notebook/kernel **current working directory** set to the parent of `config/` (usually the repo root or `demos/`).

The next cell installs `gdown` only if needed and, when neither `demos/config` nor `./config` is present, downloads that Drive folder into `./config` automatically (typical **Colab** flow without uploading YAML by hand).


In [ ]:
GOVERNANCE_DRIVE_FOLDER_ID = "17gH3w247NtejX4911wFGXQ_8dPUVlHhe"
GOVERNANCE_DRIVE_FOLDER = (
    f"https://drive.google.com/drive/folders/{GOVERNANCE_DRIVE_FOLDER_ID}?usp=sharing"
)


def _find_governance_root(search_under: Path):
    """Return a directory that contains governance/defaults.yaml."""
    marker = Path("governance") / "defaults.yaml"
    if (search_under / marker).is_file():
        return search_under
    if not search_under.is_dir():
        return None
    for child in search_under.iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    return None


def ensure_config_from_google_drive() -> None:
    """Use bundled demos/config, or ./config, or download Drive folder into ./config."""
    import shutil
    import subprocess
    import sys

    marker = Path("governance") / "defaults.yaml"
    cwd = Path.cwd()
    if (cwd / "demos" / "config" / marker).is_file():
        print("✅ Using bundled demos/config from the repository.")
        return
    if (cwd / "config" / marker).is_file():
        print("✅ Using ./config (e.g. from Google Drive download).")
        return

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

    dl = cwd / "_governance_config_download"
    if dl.exists():
        shutil.rmtree(dl)
    dl.mkdir(parents=True)
    gdown.download_folder(GOVERNANCE_DRIVE_FOLDER, output=str(dl), quiet=True, use_cookies=False)

    root = _find_governance_root(dl)
    if root is None:
        shutil.rmtree(dl, ignore_errors=True)
        raise FileNotFoundError(
            "gdown did not produce governance/defaults.yaml. Download the folder manually from: "
            + GOVERNANCE_DRIVE_FOLDER
        )

    out = cwd / "config"
    if out.exists():
        shutil.rmtree(out)
    shutil.move(str(root), str(out))
    shutil.rmtree(dl, ignore_errors=True)
    print(f"✅ Downloaded governance/skills YAML into {out} (from Google Drive).")


ensure_config_from_google_drive()


✅ Downloaded governance/skills YAML into /content/config (from Google Drive).


In [ ]:
ToolCategory = Literal["discovery", "connection", "execution", "other"]

# Registry for MCP tool categories discovered at runtime
TOOL_CATEGORY_REGISTRY: Dict[str, ToolCategory] = {}
DISCOVERED_TOOL_NAMES: Set[str] = set()


def categorize_tool(tool_name: str) -> ToolCategory:
    """Categorize tools using registry first, then safe defaults."""
    name = tool_name.upper()
    if name in TOOL_CATEGORY_REGISTRY:
        return TOOL_CATEGORY_REGISTRY[name]

    discovery_tools = {
        "EXA_WEB_SEARCH",
        "EXA_CODE_SEARCH",
        "SERPAPI_LIST_TOOLS",
    }
    connection_tools = {"CONNECT_SERPAPI"}
    execution_prefixes = ("EXECUTE_", "RUN_", "SEND_", "POST_", "CREATE_", "UPDATE_", "DELETE_")

    if name in discovery_tools:
        return "discovery"
    if name in connection_tools:
        return "connection"
    if name == "SEARCH":
        return "execution"
    if name.startswith(execution_prefixes):
        return "execution"
    if "SERPAPI" in name:
        return "execution"
    return "other"


def register_tool_categories(tool_names: List[str]) -> None:
    """Register MCP tool categories based on a curated allowlist."""
    for tool in tool_names:
        if not tool:
            continue
        name = tool.upper()
        DISCOVERED_TOOL_NAMES.add(name)
        if name == "SERPAPI_LIST_TOOLS":
            TOOL_CATEGORY_REGISTRY[name] = "discovery"
        elif name == "SEARCH":
            TOOL_CATEGORY_REGISTRY[name] = "execution"
        elif "SERPAPI" in name:
            TOOL_CATEGORY_REGISTRY[name] = "execution"


class ToolAccessLevel(Enum):
    """Access levels for tools."""
    ALLOWED = "allowed"
    RESTRICTED = "restricted"
    FORBIDDEN = "forbidden"


@dataclass
class ToolPolicy:
    """Policy for a specific tool or tool category."""
    pattern: str
    access_level: ToolAccessLevel
    max_calls_per_task: int = 10
    description: str = ""


@dataclass
class GovernancePolicy:
    """Complete governance policy for A2A-MCP access."""
    name: str
    tool_policies: List[ToolPolicy]

    # Global budgets
    max_total_calls_per_task: int = 20
    max_parallel_calls: int = 3
    max_runtime_seconds: int = 300
    mcp_call_timeout_seconds: int = 20

    # Connection management
    allow_connection_creation: bool = False
    connection_and_execution_in_same_run: bool = False

    # Artifact filtering
    allowed_artifact_types: Set[str] = field(default_factory=lambda: {
        "policy_decision", "tool_call_log", "approval_log", "approval_request", "budget_stats", "result_summary", "skill_denial"
    })

    def is_tool_allowed(self, tool_name: str) -> tuple[bool, ToolAccessLevel, Optional[ToolPolicy]]:
        """Check if a tool is allowed by this policy."""
        for policy in self.tool_policies:
            if re.search(policy.pattern, tool_name, re.IGNORECASE):
                return policy.access_level != ToolAccessLevel.FORBIDDEN, policy.access_level, policy
        return False, ToolAccessLevel.FORBIDDEN, None



def resolve_config_dir() -> Path:
    """Locate config root: GOVERNANCE_CONFIG_DIR, then ./config, then demos/config."""
    marker = Path("governance") / "defaults.yaml"
    env = os.getenv("GOVERNANCE_CONFIG_DIR", "").strip()
    if env:
        p = Path(env).expanduser()
        if not p.is_absolute():
            p = (Path.cwd() / p).resolve()
        else:
            p = p.resolve()
        if (p / marker).is_file():
            return p
        print(f"⚠️ GOVERNANCE_CONFIG_DIR={env!r} missing {marker}; trying ./config and demos/config")
    for base in (Path.cwd(), Path.cwd() / "demos"):
        p = (base / "config").resolve()
        if (p / marker).is_file():
            return p
    return (Path.cwd() / "demos" / "config").resolve()


CONFIG_DIR = resolve_config_dir()


@dataclass
class SkillConstraints:
    """PRP-style skill: narrative fields + machine-enforced lists (narrowing only)."""
    name: str = ""
    objective: str = ""
    constraints_text: str = ""
    forbidden_argument_substrings: List[str] = field(default_factory=list)


def governance_policy_from_dict(data: Dict[str, Any]) -> GovernancePolicy:
    """Build GovernancePolicy from a plain dict (e.g. after OmegaConf resolve)."""
    rows = data.get("tool_policies") or []
    tool_policies: List[ToolPolicy] = []
    for row in rows:
        tool_policies.append(
            ToolPolicy(
                pattern=str(row["pattern"]),
                access_level=ToolAccessLevel(str(row["access_level"])),
                max_calls_per_task=int(row.get("max_calls_per_task", 10)),
                description=str(row.get("description", "")),
            )
        )
    allowed = data.get("allowed_artifact_types")
    allowed_set: Set[str] = set(allowed) if allowed is not None else {
        "policy_decision", "tool_call_log", "approval_log", "approval_request",
        "budget_stats", "result_summary", "skill_denial",
    }
    return GovernancePolicy(
        name=str(data["name"]),
        tool_policies=tool_policies,
        max_total_calls_per_task=int(data.get("max_total_calls_per_task", 20)),
        max_parallel_calls=int(data.get("max_parallel_calls", 3)),
        max_runtime_seconds=int(data.get("max_runtime_seconds", 300)),
        mcp_call_timeout_seconds=int(data.get("mcp_call_timeout_seconds", 20)),
        allow_connection_creation=bool(data.get("allow_connection_creation", False)),
        connection_and_execution_in_same_run=bool(data.get("connection_and_execution_in_same_run", False)),
        allowed_artifact_types=allowed_set,
    )


def load_governance_merged(defaults_path: Path, profile_path: Path) -> GovernancePolicy:
    base = OmegaConf.load(str(defaults_path))
    prof = OmegaConf.load(str(profile_path))
    merged = OmegaConf.merge(base, prof)
    resolved: Dict[str, Any] = OmegaConf.to_container(merged, resolve=True)
    return governance_policy_from_dict(resolved)


def load_skill_yaml(path: Path) -> SkillConstraints:
    raw = OmegaConf.load(str(path))
    data: Dict[str, Any] = OmegaConf.to_container(raw, resolve=True)
    return SkillConstraints(
        name=str(data.get("name", "") or ""),
        objective=str(data.get("objective", "") or ""),
        constraints_text=str(data.get("constraints", "") or ""),
        forbidden_argument_substrings=[str(x) for x in (data.get("forbidden_argument_substrings") or [])],
    )


def skill_constraints_violation(skill: Optional[SkillConstraints], arguments: Dict[str, Any]) -> Optional[str]:
    """Return a denial reason if a forbidden substring appears in serialized arguments."""
    if skill is None or not skill.forbidden_argument_substrings:
        return None
    blob = json.dumps(arguments, sort_keys=True, default=str).lower()
    for sub in skill.forbidden_argument_substrings:
        if sub.lower() in blob:
            return f"Skill {skill.name!r}: forbidden substring {sub!r} in arguments"
    return None


# Policy 1: Discovery only — loaded from OmegaConf-merge (YAML under config root / governance/)
DISCOVERY_ONLY_POLICY = load_governance_merged(
    CONFIG_DIR / "governance" / "defaults.yaml",
    CONFIG_DIR / "governance" / "discovery_only.yaml",
)

RESEARCH_STRICT_SKILL = load_skill_yaml(CONFIG_DIR / "skills" / "research_strict.yaml")

# Policy 2: Restricted SerpApi execution (approval required)
RESTRICTED_SERPAPI_POLICY = GovernancePolicy(
    name="Restricted SerpApi",
    tool_policies=[
        ToolPolicy(r"EXA_WEB_SEARCH", ToolAccessLevel.ALLOWED, 3, "Exa web search"),
        ToolPolicy(r"EXA_CODE_SEARCH", ToolAccessLevel.FORBIDDEN, 0, "Exa code search blocked"),
        ToolPolicy(r"SERPAPI_LIST_TOOLS", ToolAccessLevel.ALLOWED, 2, "SerpApi MCP tool listing"),
        ToolPolicy(r"^SEARCH$", ToolAccessLevel.RESTRICTED, 1, "SerpApi search tool requires approval"),
        ToolPolicy(r"SERPAPI_.*", ToolAccessLevel.RESTRICTED, 1, "SerpApi tools require approval"),
        ToolPolicy(r".*", ToolAccessLevel.FORBIDDEN, 0, "Block all others"),
    ],
    max_total_calls_per_task=6,
    max_runtime_seconds=180,
)

# Policy 3: Budget exhaustion test
BUDGET_LIMITED_POLICY = GovernancePolicy(
    name="Budget Limited",
    tool_policies=[
        ToolPolicy(r"EXA_WEB_SEARCH", ToolAccessLevel.ALLOWED, 3, "Exa web search - limited"),
        ToolPolicy(r"EXA_CODE_SEARCH", ToolAccessLevel.RESTRICTED, 1, "Exa code search requires approval"),
        ToolPolicy(r"SERPAPI_LIST_TOOLS", ToolAccessLevel.FORBIDDEN, 0, "SerpApi tools disabled for budget test"),
        ToolPolicy(r".*", ToolAccessLevel.FORBIDDEN, 0, "Block all others"),
    ],
    max_total_calls_per_task=3,
    max_runtime_seconds=60,
)

# Policy 4: Separation demo (connection then execution blocked)
SEPARATION_POLICY = GovernancePolicy(
    name="Connection Separation",
    tool_policies=[
        ToolPolicy(r"SERPAPI_LIST_TOOLS", ToolAccessLevel.ALLOWED, 1, "Discovery step"),
        ToolPolicy(r"^SEARCH$", ToolAccessLevel.ALLOWED, 1, "Execution step"),
        ToolPolicy(r"SERPAPI_.*", ToolAccessLevel.ALLOWED, 1, "Execution step"),
        ToolPolicy(r".*", ToolAccessLevel.FORBIDDEN, 0, "Block all others"),
    ],
    max_total_calls_per_task=2,
    max_runtime_seconds=60,
    allow_connection_creation=True,
    connection_and_execution_in_same_run=False,
)

# Policy 5: HITL approval demo (code search allowed only with approval)
HITL_POLICY = GovernancePolicy(
    name="HITL Code Search",
    tool_policies=[
        ToolPolicy(r"EXA_WEB_SEARCH", ToolAccessLevel.ALLOWED, 2, "Exa web search"),
        ToolPolicy(r"EXA_CODE_SEARCH", ToolAccessLevel.RESTRICTED, 1, "Exa code search requires approval"),
        ToolPolicy(r".*", ToolAccessLevel.FORBIDDEN, 0, "Block all others"),
    ],
    max_total_calls_per_task=2,
    max_runtime_seconds=60,
)

print(f"✅ Policies defined:")
print(f"   - {DISCOVERY_ONLY_POLICY.name}")
print(f"   - {RESTRICTED_SERPAPI_POLICY.name}")
print(f"   - {BUDGET_LIMITED_POLICY.name}")
print(f"   - {SEPARATION_POLICY.name}")
print(f"   - {HITL_POLICY.name}")
print(f"✅ DISCOVERY_ONLY_POLICY from YAML: {DISCOVERY_ONLY_POLICY.name} "
      f"(max_runtime={DISCOVERY_ONLY_POLICY.max_runtime_seconds}s)")
print(f"✅ PRP skill loaded: {RESEARCH_STRICT_SKILL.name!r}")


✅ Policies defined:
   - Discovery Only
   - Restricted SerpApi
   - Budget Limited
   - Connection Separation
   - HITL Code Search
✅ DISCOVERY_ONLY_POLICY from YAML: Discovery Only (max_runtime=120s)
✅ PRP skill loaded: 'research_strict'


In [ ]:
@dataclass
class BudgetTracker:
    """Tracks resource usage and enforces connection separation."""
    policy: GovernancePolicy
    start_time: datetime = field(default_factory=datetime.now)
    total_calls: int = 0
    active_calls: int = 0
    tool_call_counts: Dict[str, int] = field(default_factory=dict)
    did_connection_step: bool = False
    did_execution_step: bool = False
    paused_seconds: float = 0.0
    paused_at: Optional[datetime] = None
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False, repr=False)

    def pause_timer(self) -> None:
        """Pause runtime budget (e.g., while waiting for human input)."""
        with self._lock:
            if self.paused_at is None:
                self.paused_at = datetime.now()

    def resume_timer(self) -> None:
        """Resume runtime budget after a pause."""
        with self._lock:
            if self.paused_at is not None:
                self.paused_seconds += (datetime.now() - self.paused_at).total_seconds()
                self.paused_at = None

    def _elapsed_seconds(self) -> float:
        """Elapsed seconds excluding pauses."""
        with self._lock:
            elapsed = (datetime.now() - self.start_time).total_seconds()
            if self.paused_at is not None:
                elapsed -= (datetime.now() - self.paused_at).total_seconds()
            return max(0.0, elapsed - self.paused_seconds)

    def can_call_tool(self, tool_name: str, tool_category: ToolCategory) -> tuple[bool, str]:
        """Check if a tool call is allowed given current budget and separation rules."""
        # Check runtime
        elapsed = self._elapsed_seconds()
        if elapsed > self.policy.max_runtime_seconds:
            return False, f"Runtime exceeded: {elapsed:.1f}s > {self.policy.max_runtime_seconds}s"

        # Check total calls
        if self.total_calls >= self.policy.max_total_calls_per_task:
            return False, f"Total call limit: {self.total_calls} >= {self.policy.max_total_calls_per_task}"

        # Check parallel calls
        if self.active_calls >= self.policy.max_parallel_calls:
            return False, f"Parallel limit: {self.active_calls} >= {self.policy.max_parallel_calls}"

        # Check tool-specific policy
        allowed, access_level, tool_policy = self.policy.is_tool_allowed(tool_name)
        if not allowed:
            return False, f"Tool forbidden by policy"

        if tool_policy:
            tool_calls = self.tool_call_counts.get(tool_name, 0)
            if tool_calls >= tool_policy.max_calls_per_task:
                return False, f"Tool call limit: {tool_calls} >= {tool_policy.max_calls_per_task}"

        # Check connection/execution separation
        if not self.policy.connection_and_execution_in_same_run:
            if tool_category == "connection" and self.did_execution_step:
                return False, "Connection after execution: separation policy violated"
            if tool_category == "execution" and self.did_connection_step:
                return False, "Execution after connection: separation policy violated"

        return True, ""

    def record_call_start(self, tool_name: str, tool_category: ToolCategory):
        """Record the start of a tool call."""
        with self._lock:
            self.total_calls += 1
            self.active_calls += 1
            self.tool_call_counts[tool_name] = self.tool_call_counts.get(tool_name, 0) + 1

            if tool_category == "connection":
                self.did_connection_step = True
            elif tool_category == "execution":
                self.did_execution_step = True

    def record_call_end(self):
        """Record the end of a tool call."""
        with self._lock:
            self.active_calls = max(0, self.active_calls - 1)

    def get_stats(self) -> Dict[str, Any]:
        """Get current budget statistics."""
        elapsed = self._elapsed_seconds()
        return {
            "elapsed_seconds": round(elapsed, 2),
            "total_calls": self.total_calls,
            "active_calls": self.active_calls,
            "remaining_calls": self.policy.max_total_calls_per_task - self.total_calls,
            "remaining_runtime": round(self.policy.max_runtime_seconds - elapsed, 2),
            "tool_calls": dict(self.tool_call_counts),
            "did_connection_step": self.did_connection_step,
            "did_execution_step": self.did_execution_step,
        }

print("✅ Budget tracker with separation tracking implemented")

✅ Budget tracker with separation tracking implemented


## 4. Argument Validation

Validate tool arguments before execution to prevent runaway behavior.

In [ ]:
class ArgumentValidator:
    """Validates tool arguments against gates."""

    @staticmethod
    def validate_exa_search(args: Dict[str, Any]) -> tuple[bool, str]:
        """Validate Exa search arguments."""
        query = args.get("query", "")
        if not query or not isinstance(query, str):
            return False, "Query must be a non-empty string"
        if len(query) > 500:
            return False, f"Query too long: {len(query)} > 500"

        num_results = args.get("num_results", 5)
        if not isinstance(num_results, int) or num_results < 1 or num_results > 5:
            return False, "num_results must be between 1 and 5"

        return True, ""

    @staticmethod
    def validate_connect_serpapi(args: Dict[str, Any]) -> tuple[bool, str]:
        """No args expected for connection."""
        if args:
            return False, "CONNECT_SERPAPI takes no arguments"
        return True, ""

    @staticmethod
    def validate_serpapi_list_tools(args: Dict[str, Any]) -> tuple[bool, str]:
        """No args expected for tool listing."""
        if args:
            return False, "SERPAPI_LIST_TOOLS takes no arguments"
        return True, ""

    @staticmethod
    def validate_serpapi_search(args: Dict[str, Any]) -> tuple[bool, str]:
        """Validate SerpApi search arguments."""
        query = args.get("q") or args.get("query")
        if not query or not isinstance(query, str):
            return False, "Search query must be provided as 'q' or 'query'"
        if len(query) > 500:
            return False, f"Query too long: {len(query)} > 500"

        num = args.get("num") or args.get("num_results")
        if num is not None:
            try:
                num_val = int(num)
            except Exception:
                return False, "num must be an integer"
            if num_val < 1 or num_val > 10:
                return False, "num must be between 1 and 10"

        start = args.get("start")
        if start is not None:
            try:
                start_val = int(start)
            except Exception:
                return False, "start must be an integer"
            if start_val < 0 or start_val > 50:
                return False, "start must be between 0 and 50"

        location = args.get("location")
        if isinstance(location, str) and len(location) > 80:
            return False, "location too long"

        safe = args.get("safe")
        if safe is not None and str(safe).lower() not in {"active", "off", "on", "true", "false"}:
            return False, "safe must be one of active/off/on/true/false"

        return True, ""

    def validate(self, tool_name: str, arguments: Dict[str, Any]) -> tuple[bool, str]:
        """Validate arguments for a specific tool."""
        if tool_name == "CONNECT_SERPAPI":
            return self.validate_connect_serpapi(arguments)
        if tool_name in ["EXA_WEB_SEARCH", "EXA_CODE_SEARCH"]:
            return self.validate_exa_search(arguments)
        if tool_name == "SERPAPI_LIST_TOOLS":
            return self.validate_serpapi_list_tools(arguments)
        if tool_name.upper().startswith("SERPAPI_") or "SERPAPI" in tool_name.upper():
            return self.validate_serpapi_search(arguments)
        return True, ""

print("✅ Argument validation gates implemented")

✅ Argument validation gates implemented


## 5. Approval Workflow

Handle approval requests for restricted tools using LangGraph interrupts, integrated with A2A events and checkpointing.

In [ ]:
# Approval is handled via LangGraph interrupts in check_approval_node.
# No standalone ApprovalWorkflow class is used in this notebook.
print("✅ Approval uses LangGraph interrupts")

✅ Approval uses LangGraph interrupts


## 6. MCP Session Manager

Manages MCP client sessions with proper lifecycle.

In [ ]:
class MCPSessionManager:
    """Manages MCP client sessions."""

    def __init__(self, mcp_url: str, headers: Optional[Dict[str, str]] = None):
        self.mcp_url = mcp_url
        self.headers = headers or {}
        self.exit_stack: Optional[AsyncExitStack] = None
        self.session: Optional[ClientSession] = None

    def is_connected(self) -> bool:
        """Check if MCP session is connected."""
        return self.session is not None

    async def connect(self):
        """Establish MCP connection (idempotent)."""
        if self.session:
            return
        self.exit_stack = AsyncExitStack()

        transport = await self.exit_stack.enter_async_context(
            streamablehttp_client(self.mcp_url, headers=self.headers)
        )

        if isinstance(transport, tuple):
            read_stream, write_stream = transport[0], transport[1]
        else:
            read_stream = transport.read_stream
            write_stream = transport.write_stream

        self.session = await self.exit_stack.enter_async_context(
            ClientSession(read_stream, write_stream)
        )
        await self.session.initialize()

    async def ensure_connected(self):
        """Connect on demand if needed."""
        if not self.session:
            await self.connect()

    async def disconnect(self):
        """Close MCP connection."""
        if self.exit_stack:
            await self.exit_stack.aclose()
            self.exit_stack = None
            self.session = None

    async def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Any:
        """Call an MCP tool."""
        await self.ensure_connected()
        result = await self.session.call_tool(tool_name, arguments)
        return result

    async def list_tools(self) -> List[str]:
        """List available MCP tools."""
        await self.ensure_connected()
        tools_resp = await self.session.list_tools()
        tools = getattr(tools_resp, "tools", None) or []
        return [getattr(t, "name", "") for t in tools if getattr(t, "name", "")]

print("✅ MCP session manager implemented")

✅ MCP session manager implemented


## 7. governed_call - Central Enforcement Point

Single function that enforces all policies and executes privileged tools (MCP or direct). This is the only place that can perform external actions.

In [ ]:
def _extract_serpapi_tool_names(tool_names: List[str]) -> List[str]:
    """Normalize MCP tool names for SerpApi discovery."""
    unique = [name for name in tool_names if name]
    return list(dict.fromkeys(unique))


async def governed_call(
    request: ToolRequest,
    policy: GovernancePolicy,
    budget: BudgetTracker,
    mcp_session: MCPSessionManager,
    validator: ArgumentValidator,
    updater: Optional[TaskUpdater] = None,
    skill_constraints: Optional[SkillConstraints] = None,
) -> ToolResult:
    """Execute governed MCP tool call with full enforcement."""

    tool_name_raw = request.tool_name
    tool_name = tool_name_raw.upper()
    tool_category = categorize_tool(tool_name)

    async def add_artifact(name: str, text: str) -> None:
        if updater and name in policy.allowed_artifact_types:
            await updater.add_artifact([Part(root=TextPart(text=text))], name=name)

    # Step 1: Policy match
    allowed, access_level, tool_policy = policy.is_tool_allowed(tool_name)
    policy_match = tool_policy.description if tool_policy else "No match"
    policy_pattern = tool_policy.pattern if tool_policy else None

    if not allowed:
        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason="Tool forbidden by policy",
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )

        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Blocked: {tool_name} - forbidden by policy", "", "")
            )
            await add_artifact("policy_decision", result.to_json())

        return result

    if tool_category == "connection" and not policy.allow_connection_creation:
        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason="Connection creation forbidden by policy",
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )
        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Blocked: {tool_name} - connection creation forbidden", "", "")
            )
            await add_artifact("policy_decision", result.to_json())
        return result

    if tool_name not in ["EXA_WEB_SEARCH", "EXA_CODE_SEARCH", "CONNECT_SERPAPI", "SERPAPI_LIST_TOOLS"]:
        if DISCOVERED_TOOL_NAMES and tool_name not in DISCOVERED_TOOL_NAMES:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason="Tool not in discovered MCP allowlist",
                policy_match=policy_match,
                policy_pattern=policy_pattern,
                budget_stats_snapshot=budget.get_stats()
            )
            if updater:
                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message(f"❌ Blocked: {tool_name_raw} - not discovered", "", "")
                )
                await add_artifact("policy_decision", result.to_json())
            return result

    # Step 2 & 3: Budget and separation checks
    can_call, budget_reason = budget.can_call_tool(tool_name, tool_category)
    if not can_call:
        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason=budget_reason,
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )

        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Blocked: {budget_reason}", "", "")
            )
            await add_artifact("budget_stats", json.dumps(budget.get_stats(), indent=2))
            await add_artifact("policy_decision", result.to_json())

        return result

    # Step 4: Argument validation
    valid, validation_msg = validator.validate(tool_name, request.arguments)
    if not valid:
        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason=f"Invalid arguments: {validation_msg}",
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )

        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Blocked: {validation_msg}", "", "")
            )
            await add_artifact("policy_decision", result.to_json())

        return result

    # Step 4b: PRP / skill constraints (narrows allowed work)
    skill_reason = skill_constraints_violation(skill_constraints, request.arguments)
    if skill_reason:
        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason=f"Skill constraint: {skill_reason}",
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )
        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Blocked: {skill_reason}", "", "")
            )
            await add_artifact("skill_denial", json.dumps({"reason": skill_reason, "skill": skill_constraints.name if skill_constraints else None}, indent=2))
            await add_artifact("policy_decision", result.to_json())
        return result

    # Step 5: Approval is handled by LangGraph checkpointing (no-op here)
    # Step 6: Execute tool
    budget.record_call_start(tool_name, tool_category)

    if updater:
        await updater.update_status(
            TaskState.working,
            new_agent_text_message(f"🔧 Executing {tool_name}", "", "")
        )

    try:
        extracted_tool_slugs = []
        output_str = ""
        raw_output_ref = "full_output"

        async def _ensure_mcp_connected():
            if not mcp_session.is_connected():
                budget.did_connection_step = True
                if updater:
                    await updater.update_status(
                        TaskState.working,
                        new_agent_text_message("🔌 Connecting to MCP...", "", "")
                    )
                await mcp_session.ensure_connected()

        if tool_name in ["EXA_WEB_SEARCH", "EXA_CODE_SEARCH"]:
            exa = Exa(api_key=EXA_API_KEY)
            query = request.arguments.get("query", "")
            num_results = request.arguments.get("num_results", 5)
            search_type = "keyword" if tool_name == "EXA_CODE_SEARCH" else "auto"
            exa_result = await asyncio.wait_for(
                asyncio.to_thread(exa.search, query, num_results=num_results, type=search_type),
                timeout=policy.mcp_call_timeout_seconds
            )
            items = getattr(exa_result, "results", None) or []
            preview = [
                {
                    "title": getattr(item, "title", ""),
                    "url": getattr(item, "url", "")
                }
                for item in items[:3]
            ]
            output_str = json.dumps({"results": preview}, indent=2)
            raw_output_ref = "exa_results"
        elif tool_name == "CONNECT_SERPAPI":
            await _ensure_mcp_connected()
            output_str = "Connected to MCP"
            raw_output_ref = "mcp_connection"
        elif tool_name == "SERPAPI_LIST_TOOLS":
            await _ensure_mcp_connected()
            tool_names = await mcp_session.list_tools()
            register_tool_categories(tool_names)
            extracted_tool_slugs = _extract_serpapi_tool_names(tool_names)
            output_str = json.dumps({"tools": extracted_tool_slugs[:20]}, indent=2)
            raw_output_ref = "serpapi_tools"
        else:
            await _ensure_mcp_connected()
            mcp_result = await asyncio.wait_for(
                mcp_session.call_tool(tool_name_raw, request.arguments),
                timeout=policy.mcp_call_timeout_seconds
            )
            output_str = str(mcp_result)[:500]
            raw_output_ref = "mcp_output"

        result = ToolResult(
            tool_name=tool_name,
            allowed=True,
            decision_reason="Executed successfully",
            output_summary=output_str,
            raw_output_ref=raw_output_ref,
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            extracted_tool_slugs=extracted_tool_slugs,
            budget_stats_snapshot=budget.get_stats()
        )

        # Step 7: Emit tool_call_log
        if updater:
            tool_log = json.dumps({
                "tool_name": tool_name,
                "category": tool_category,
                "arguments": str(request.arguments)[:200],
                "output_summary": output_str,
                "extracted_tool_slugs": extracted_tool_slugs,
                "timestamp": datetime.now().isoformat()
            }, indent=2)

            await add_artifact("tool_call_log", tool_log)

        await add_artifact("policy_decision", result.to_json())
        return result

    except Exception as e:

        result = ToolResult(
            tool_name=tool_name,
            allowed=False,
            decision_reason=f"Execution error: {str(e)[:200]}",
            policy_match=policy_match,
            policy_pattern=policy_pattern,
            budget_stats_snapshot=budget.get_stats()
        )

        if updater:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"❌ Execution error: {str(e)[:200]}", "", "")
            )
            await add_artifact("policy_decision", result.to_json())

        return result
    finally:
        budget.record_call_end()

print("✅ governed_call enforcement function implemented")


✅ governed_call enforcement function implemented


In [ ]:
class AgentState(LangGraphTypedDict):
    """State for the LangGraph agent."""
    query: str
    policy_summary: str
    messages: List[Any]
    tool_results: List[ToolResult]
    tools_used: int
    final_answer: Optional[str]
    pending_request: Optional[ToolRequest]
    needs_approval: bool  # Reserved (legacy)
    approval_granted: Optional[bool]  # Set by interrupt resume
    discovered_tool_slugs: List[str]
    did_connection_step: bool
    did_execution_step: bool
    use_llm_proposal: bool
    force_separation_demo: bool


def create_governed_agent(
    policy: GovernancePolicy,
    budget: BudgetTracker,
    mcp_session: MCPSessionManager,
    validator: ArgumentValidator,
    updater: Optional[TaskUpdater],
    llm: ChatOpenAI,
    skill_constraints: Optional[SkillConstraints] = None,
) -> StateGraph:
    """Create LangGraph agent with checkpointing for approval workflows."""


    def _pick_serpapi_tool(tool_slugs: List[str]) -> Optional[str]:
        if not tool_slugs:
            return None
        preferred = ["GOOGLE", "NEWS", "SEARCH", "WEB"]
        for slug in tool_slugs:
            if any(kw in slug.upper() for kw in preferred):
                return slug
        return tool_slugs[0]

    async def add_artifact(name: str, text: str) -> None:
        if updater and name in policy.allowed_artifact_types:
            await updater.add_artifact([Part(root=TextPart(text=text))], name=name)

    async def propose_request_node(state: AgentState) -> AgentState:
        """Agent proposes next ToolRequest or FinalAnswer."""
        tools_used = state.get("tools_used", 0)
        tool_results = state.get("tool_results", [])
        discovered_tool_slugs = state.get("discovered_tool_slugs", [])
        query_text = state.get("query", "")
        query_lower = query_text.lower()

        code_query = any(
            kw in query_lower
            for kw in [
                "stack overflow", "stackoverflow", "code", "coding", "programming",
                "bug", "error", "exception", "traceback", "syntax", "snippet"
            ]
        )

        # First call: LLM proposes ToolRequest if enabled
        if tools_used == 0:
            if state.get("force_separation_demo"):
                return {
                    **state,
                    "pending_request": ToolRequest(
                        tool_name="SERPAPI_LIST_TOOLS",
                        arguments={},
                        rationale="Separation demo: discovery step"
                    ),
                    "needs_approval": False,
                    "discovered_tool_slugs": []
                }
            if state.get("use_llm_proposal"):
                prompt = (
                    "Return JSON with keys tool_name and arguments. "
                    "Choose EXA_WEB_SEARCH for general web queries, "
                    "or EXA_CODE_SEARCH for coding/StackOverflow queries. "
                    f"Query: {query_text}"
                )
                try:
                    llm_resp = await asyncio.to_thread(llm.invoke, [HumanMessage(content=prompt)])
                    parsed = json.loads(llm_resp.content)
                    model = ToolRequestModel.model_validate(parsed)
                    tool_name = model.tool_name
                    arguments = model.arguments
                except Exception:
                    tool_name = None
                    arguments = None

                default_tool = "EXA_CODE_SEARCH" if code_query else "EXA_WEB_SEARCH"
                if not isinstance(tool_name, str):
                    tool_name = default_tool
                if not isinstance(arguments, dict):
                    arguments = {"query": query_text, "num_results": 5}

                tool_allowed, _, _ = policy.is_tool_allowed(tool_name)
                if not tool_allowed:
                    fallback_allowed, _, _ = policy.is_tool_allowed(default_tool)
                    if not fallback_allowed:
                        return {
                            **state,
                            "final_answer": "No allowed tools for this request under current policy.",
                            "pending_request": None,
                            "needs_approval": False,
                            "discovered_tool_slugs": []
                        }
                    tool_name = default_tool
                    arguments = {"query": query_text, "num_results": 5}

                return {
                    **state,
                    "pending_request": ToolRequest(
                        tool_name=tool_name,
                        arguments=arguments,
                        rationale="LLM proposed ToolRequest"
                    ),
                    "needs_approval": False,
                    "discovered_tool_slugs": []
                }
            tool_name = "EXA_CODE_SEARCH" if code_query else "EXA_WEB_SEARCH"
            rationale = "Attempting code search (expected to be blocked)" if code_query else "Normal web search"
            return {
                **state,
                "pending_request": ToolRequest(
                    tool_name=tool_name,
                    arguments={"query": query_text, "num_results": 5},
                    rationale=rationale
                ),
                "needs_approval": False,
                "discovered_tool_slugs": []
            }

        if tool_results:
            last_result = tool_results[-1]

            # If code search was blocked, fall back to web search once
            if last_result.tool_name == "EXA_CODE_SEARCH" and not last_result.allowed:
                already_tried_web = any(tr.tool_name == "EXA_WEB_SEARCH" for tr in tool_results)
                web_allowed, _, _ = policy.is_tool_allowed("EXA_WEB_SEARCH")
                if web_allowed and not already_tried_web:
                    return {
                        **state,
                        "pending_request": ToolRequest(
                            tool_name="EXA_WEB_SEARCH",
                            arguments={"query": query_text, "num_results": 5},
                            rationale="Fallback to web search after code search blocked"
                        ),
                        "needs_approval": False,
                        "discovered_tool_slugs": discovered_tool_slugs
                    }
                return {
                    **state,
                    "final_answer": "Code search blocked by policy. Human approval required for exceptions.",
                    "pending_request": None,
                    "needs_approval": False,
                    "discovered_tool_slugs": discovered_tool_slugs
                }

            # Separation demo: after discovery, try execution
            if state.get("force_separation_demo") and last_result.tool_name == "SERPAPI_LIST_TOOLS" and last_result.allowed:
                discovered_tool_slugs = last_result.extracted_tool_slugs
                candidate = _pick_serpapi_tool(discovered_tool_slugs) or "SEARCH"
                return {
                    **state,
                    "pending_request": ToolRequest(
                        tool_name=candidate,
                        arguments={"q": query_text},
                        rationale="Separation demo: execute after discovery"
                    ),
                    "needs_approval": False,
                    "discovered_tool_slugs": discovered_tool_slugs
                }

            # After Exa web search, continue budget loop or optionally list SerpApi tools
            if last_result.tool_name == "EXA_WEB_SEARCH" and last_result.allowed:
                if policy.name == "Budget Limited" and tools_used < policy.max_total_calls_per_task:
                    return {
                        **state,
                        "pending_request": ToolRequest(
                            tool_name="EXA_WEB_SEARCH",
                            arguments={"query": f"{query_text} (refine {tools_used})", "num_results": 3},
                            rationale="Iterative web search for budget exhaustion"
                        ),
                        "needs_approval": False,
                        "discovered_tool_slugs": discovered_tool_slugs
                    }
                if "serpapi" in query_lower or "list tools" in query_lower:
                    return {
                        **state,
                        "pending_request": ToolRequest(
                            tool_name="SERPAPI_LIST_TOOLS",
                            arguments={},
                            rationale="Discover SerpApi MCP tools"
                        ),
                        "needs_approval": False,
                        "discovered_tool_slugs": discovered_tool_slugs
                    }
                return {
                    **state,
                    "final_answer": "Web search completed.",
                    "pending_request": None,
                    "needs_approval": False,
                    "discovered_tool_slugs": discovered_tool_slugs
                }

            # After listing SerpApi tools, attempt a concrete SerpApi tool
            if last_result.tool_name == "SERPAPI_LIST_TOOLS" and last_result.allowed:
                discovered_tool_slugs = last_result.extracted_tool_slugs
                candidate = _pick_serpapi_tool(discovered_tool_slugs)
                if candidate:
                    return {
                        **state,
                        "pending_request": ToolRequest(
                            tool_name=candidate,
                            arguments={"q": query_text},
                            rationale="Attempting a concrete SerpApi search tool"
                        ),
                        "needs_approval": False,
                        "discovered_tool_slugs": discovered_tool_slugs
                    }

                summary = "No SerpApi tools discovered from MCP list_tools."
                return {
                    **state,
                    "final_answer": summary,
                    "pending_request": None,
                    "needs_approval": False,
                    "discovered_tool_slugs": discovered_tool_slugs
                }

            # Budget demo: keep searching until budget is exhausted
            if policy.name == "Budget Limited" and last_result.allowed:
                if tools_used < policy.max_total_calls_per_task:
                    return {
                        **state,
                        "pending_request": ToolRequest(
                            tool_name="EXA_WEB_SEARCH",
                            arguments={"query": f"{query_text} (refine {tools_used})", "num_results": 3},
                            rationale="Iterative web search for budget exhaustion"
                        ),
                        "needs_approval": False,
                        "discovered_tool_slugs": discovered_tool_slugs
                    }

        # After tool execution, provide summary and terminate
        summary = f"Tool discovery completed. {len(tool_results)} tool(s) attempted."
        if discovered_tool_slugs:
            top_tools = ", ".join(discovered_tool_slugs[:3])
            summary = f"Discovered SerpApi tools: {top_tools}. Attempted {len(tool_results)} tool(s)."
        elif tool_results and tool_results[-1].allowed:
            summary = "Completed tool calls. Check tool_call_log artifacts for details."

        return {
            **state,
            "final_answer": summary,
            "pending_request": None,
            "needs_approval": False,
            "discovered_tool_slugs": discovered_tool_slugs
        }

    async def check_approval_node(state: AgentState) -> AgentState:
        """Pause for approval using LangGraph interrupts when needed."""
        request = state.get("pending_request")
        if not request:
            return {**state, "needs_approval": False}

        tool_name = request.tool_name.upper()
        tool_category = categorize_tool(tool_name)
        budget.did_connection_step = state.get("did_connection_step", False)
        budget.did_execution_step = state.get("did_execution_step", False)

        allowed, access_level, tool_policy = policy.is_tool_allowed(tool_name)
        if not allowed:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason="Tool forbidden by policy",
                policy_match=tool_policy.description if tool_policy else "No match",
                policy_pattern=tool_policy.pattern if tool_policy else None,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "pending_request": None,
                "final_answer": f"Tool blocked: {tool_name} - forbidden by policy",
                "approval_granted": None
            }

        if tool_category == "connection" and not policy.allow_connection_creation:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason="Connection creation forbidden by policy",
                policy_match=tool_policy.description if tool_policy else "No match",
                policy_pattern=tool_policy.pattern if tool_policy else None,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "pending_request": None,
                "final_answer": f"Tool blocked: {tool_name} - connection creation forbidden",
                "approval_granted": None
            }

        can_call, budget_reason = budget.can_call_tool(tool_name, tool_category)
        if not can_call:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason=budget_reason,
                policy_match=tool_policy.description if tool_policy else "No match",
                policy_pattern=tool_policy.pattern if tool_policy else None,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "pending_request": None,
                "final_answer": f"Tool blocked: {tool_name} - {budget_reason}",
                "approval_granted": None
            }

        valid, validation_msg = validator.validate(tool_name, request.arguments)
        if not valid:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason=f"Invalid arguments: {validation_msg}",
                policy_match=tool_policy.description if tool_policy else "No match",
                policy_pattern=tool_policy.pattern if tool_policy else None,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "pending_request": None,
                "final_answer": f"Tool blocked: {tool_name} - {validation_msg}",
                "approval_granted": None
            }

        sk_reason = skill_constraints_violation(skill_constraints, request.arguments)
        if sk_reason:
            result = ToolResult(
                tool_name=tool_name,
                allowed=False,
                decision_reason=f"Skill constraint: {sk_reason}",
                policy_match=tool_policy.description if tool_policy else "No match",
                policy_pattern=tool_policy.pattern if tool_policy else None,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            await add_artifact("skill_denial", json.dumps({"reason": sk_reason, "skill": skill_constraints.name if skill_constraints else None}, indent=2))
            await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "pending_request": None,
                "final_answer": f"Tool blocked: {tool_name} - {sk_reason}",
                "approval_granted": None
            }

        if access_level != ToolAccessLevel.RESTRICTED:
            return {**state, "needs_approval": False}

        approval_payload = {
            "tool_name": request.tool_name,
            "arguments": request.arguments,
            "rationale": request.rationale,
            "timestamp": time.time()
        }
        approved = interrupt(approval_payload)
        return {**state, "approval_granted": approved, "needs_approval": False}

    async def governor_execute_node(state: AgentState) -> AgentState:
        """Execute governed_call with pending request."""
        request = state.get("pending_request")
        if not request:
            return {**state, "final_answer": "No tool request to execute", "pending_request": None}

        # Check if approval was required and granted
        approval_granted = state.get("approval_granted")
        allowed, access_level, tool_policy = policy.is_tool_allowed(request.tool_name)

        if access_level == ToolAccessLevel.RESTRICTED and approval_granted is not True:
            # Approval was denied or not provided
            result = ToolResult(
                tool_name=request.tool_name.upper(),
                allowed=False,
                decision_reason="Approval denied or not provided",
                policy_match="Restricted tool",
                policy_pattern=tool_policy.pattern if tool_policy else request.tool_name,
                budget_stats_snapshot=budget.get_stats()
            )
            tool_results = state.get("tool_results", [])
            tool_results.append(result)
            if updater:
                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message(f"❌ Blocked: {request.tool_name} - approval denied", "", "")
                )
                await add_artifact("policy_decision", result.to_json())
            return {
                **state,
                "tool_results": tool_results,
                "tools_used": state.get("tools_used", 0) + 1,
                "pending_request": None,
                "needs_approval": False,
                "approval_granted": None,
                "did_connection_step": state.get("did_connection_step", False),
                "did_execution_step": state.get("did_execution_step", False)
            }

        budget.did_connection_step = state.get("did_connection_step", False)
        budget.did_execution_step = state.get("did_execution_step", False)

        result = await governed_call(
            request, policy, budget, mcp_session,
            validator, updater, skill_constraints
        )

        tool_results = state.get("tool_results", [])
        tool_results.append(result)

        return {
            **state,
            "tool_results": tool_results,
            "tools_used": state.get("tools_used", 0) + 1,
            "pending_request": None,
            "needs_approval": False,
            "approval_granted": None,
            "did_connection_step": budget.did_connection_step,
            "did_execution_step": budget.did_execution_step
        }

    async def update_context_node(state: AgentState) -> AgentState:
        """Check if we should terminate early."""
        tool_results = state.get("tool_results", [])

        if tool_results:
            last_result = tool_results[-1]

            # If blocked by budget, terminate immediately
            if not last_result.allowed and "limit" in last_result.decision_reason.lower():
                return {
                    **state,
                    "final_answer": f"Budget limit reached: {last_result.decision_reason}",
                    "pending_request": None,
                    "needs_approval": False
                }

            # If policy blocked, terminate immediately
            if not last_result.allowed:
                return {
                    **state,
                    "final_answer": f"Tool blocked: {last_result.tool_name} - {last_result.decision_reason}",
                    "pending_request": None,
                    "needs_approval": False
                }

        # Otherwise continue to summarize
        return state

    async def cleanup_node(state: AgentState) -> AgentState:
        """Disconnect MCP session in the graph task."""
        await mcp_session.disconnect()
        return state

    def should_continue(state: AgentState) -> str:
        """Route based on state."""
        if state.get("final_answer"):
            return "cleanup"
        if state.get("pending_request"):
            return "execute"
        return "propose"

    # Build graph with checkpointer
    workflow = StateGraph(AgentState)
    workflow.add_node("propose", propose_request_node)
    workflow.add_node("check_approval", check_approval_node)
    workflow.add_node("execute", governor_execute_node)
    workflow.add_node("update", update_context_node)
    workflow.add_node("cleanup", cleanup_node)

    workflow.set_entry_point("propose")
    workflow.add_conditional_edges(
        "propose",
        should_continue,
        {"execute": "check_approval", "cleanup": "cleanup", "propose": "propose"}
    )
    workflow.add_conditional_edges(
        "check_approval",
        should_continue,
        {"execute": "execute", "cleanup": "cleanup"}
    )
    workflow.add_edge("execute", "update")
    workflow.add_conditional_edges(
        "update",
        should_continue,
        {"propose": "propose", "cleanup": "cleanup"}
    )
    workflow.add_edge("cleanup", END)

    # Add checkpointer for state persistence
    checkpointer = MemorySaver()
    return workflow.compile(checkpointer=checkpointer)

print("✅ LangGraph agent with checkpointing compiled")


✅ LangGraph agent with checkpointing compiled


In [ ]:
class GovernedMCPExecutor(AgentExecutor):
    """A2A Agent Executor with governed MCP access and checkpointing."""

    def __init__(self, policy: GovernancePolicy, mcp_url: str, mcp_headers: Dict[str, str], llm: ChatOpenAI, use_llm_proposal: bool = False, approval_on_pause: Optional[bool] = False, force_separation_demo: bool = False, notebook_mode: bool = False, skill_constraints: Optional[SkillConstraints] = None):
        super().__init__()
        self.policy = policy
        self.mcp_url = mcp_url
        self.mcp_headers = mcp_headers
        self.llm = llm
        self.thread_id = None  # Track thread for checkpointing
        self.use_llm_proposal = use_llm_proposal
        self.approval_on_pause = approval_on_pause
        self.force_separation_demo = force_separation_demo
        self.notebook_mode = notebook_mode
        self.skill_constraints = skill_constraints

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Execute governed MCP workflow with checkpointing support."""
        user_query = context.get_user_input().strip()
        task = context.current_task or new_task(context.message)
        await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)

        async def add_artifact(name: str, text: str) -> None:
            if name in self.policy.allowed_artifact_types:
                await updater.add_artifact([Part(root=TextPart(text=text))], name=name)

        # Initialize components
        budget = BudgetTracker(self.policy)
        validator = ArgumentValidator()
        mcp_session = MCPSessionManager(self.mcp_url, self.mcp_headers)

        try:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(f"🛡️ Governor active: {self.policy.name}", task.context_id, task.id)
            )

            # Connect to MCP
            await updater.update_status(
                TaskState.working,
                new_agent_text_message("🔌 MCP connection will be established on demand", task.context_id, task.id)
            )

            # Create LangGraph agent with checkpointing
            agent = create_governed_agent(
                self.policy, budget, mcp_session,
                validator, updater, self.llm, self.skill_constraints
            )

            # Run agent with checkpointing
            self.thread_id = task.id
            config = {"configurable": {"thread_id": self.thread_id}}

            initial_state = {
                "query": user_query,
                "policy_summary": self.policy.name,
                "messages": [],
                "tool_results": [],
                "tools_used": 0,
                "final_answer": None,
                "pending_request": None,
                "needs_approval": False,
                "approval_granted": None,
                "discovered_tool_slugs": [],
                "did_connection_step": False,
                "did_execution_step": False,
                "use_llm_proposal": self.use_llm_proposal,
                "force_separation_demo": self.force_separation_demo
            }

            final_state = await agent.ainvoke(initial_state, config)

            # Handle LangGraph interrupt for approval
            if "__interrupt__" in final_state:
                interrupt_payload = final_state.get("__interrupt__", [])[0].value
                await add_artifact("approval_request", json.dumps(interrupt_payload, indent=2))
                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message("⏸️  Approval required - graph paused", task.context_id, task.id)
                )
                budget.pause_timer()
                if self.approval_on_pause is None:
                    if self.notebook_mode:
                        prompt = "Approve this tool call? (y/n): "
                        answer = (await asyncio.to_thread(input, prompt)).strip().lower()
                        approved = answer in {"y", "yes"}
                        reason = "notebook input"
                    else:
                        approved = False
                        reason = "no external approval wired"
                else:
                    approved = bool(self.approval_on_pause)
                    reason = "demo approval" if approved else "demo denial"
                budget.resume_timer()
                approval_log = {
                    "tool_name": interrupt_payload.get("tool_name"),
                    "approved": approved,
                    "timestamp": time.time(),
                    "reason": reason
                }
                await add_artifact("approval_log", json.dumps(approval_log, indent=2))
                if reason == "notebook input":
                    status_reason = "notebook input"
                elif reason == "no external approval wired":
                    status_reason = "external"
                else:
                    status_reason = "demo"
                status_text = "✅ Approval granted" if approved else "❌ Approval denied"
                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message(
                        f"{status_text} ({status_reason})",
                        task.context_id,
                        task.id
                    )
                )
                final_state = await agent.ainvoke(Command(resume=approved), config)

            # Emit final result (clean allowed vs denied)
            tool_results = final_state.get("tool_results", [])
            allowed_tools = [tr.tool_name for tr in tool_results if tr.allowed]
            denied_tools = [tr.tool_name for tr in tool_results if not tr.allowed]
            allowed_outputs = []
            for tr in tool_results:
                if not tr.allowed or not tr.output_summary:
                    continue
                summary_text = tr.output_summary[:200]
                try:
                    parsed = json.loads(tr.output_summary)
                    results = parsed.get("results", []) if isinstance(parsed, dict) else []
                    if results:
                        summary_text = "; ".join(
                            f"{r.get('title', '')} ({r.get('url', '')})"
                            for r in results
                            if isinstance(r, dict)
                        )[:200]
                except Exception:
                    pass
                allowed_outputs.append({"tool": tr.tool_name, "summary": summary_text})

            transition_summary = ""
            if "EXA_CODE_SEARCH" in denied_tools and "EXA_WEB_SEARCH" in allowed_tools:
                transition_summary = "code search blocked → web search allowed"

            summary_payload = {
                "final_answer": final_state.get("final_answer", "No answer"),
                "allowed_tools": allowed_tools,
                "denied_tools": denied_tools,
                "allowed_outputs": allowed_outputs,
                "transition_summary": transition_summary
            }
            await add_artifact("result_summary", json.dumps(summary_payload, indent=2))

            # Emit final budget stats
            await add_artifact("budget_stats", json.dumps(budget.get_stats(), indent=2))

            await updater.complete()

        except Exception as e:
            await updater.update_status(
                TaskState.failed,
                new_agent_text_message(f"❌ Error: {str(e)}", task.context_id, task.id),
                final=True
            )
        finally:
            pass  # MCP disconnect handled in graph cleanup node

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Cancel execution."""
        pass

print("✅ A2A Governor executor with checkpointing implemented")


✅ A2A Governor executor with checkpointing implemented


## 10. Utility Functions

Helper functions for A2A server and client.

In [ ]:
def _text_from_message(msg):
    """Extract text from A2A message."""
    if not msg or not getattr(msg, "parts", None):
        return None
    p0 = msg.parts[0]
    root = getattr(p0, "root", None)
    return getattr(root, "text", None)


def pretty_event(evt):
    """Pretty print A2A events."""
    task, event = evt[0], evt[1]
    if event is None:
        print(f"[task] {task.id[:8]} submitted")
        return

    kind = getattr(event, "kind", None)
    if kind == "status-update":
        status = event.status
        text = _text_from_message(status.message)
        print(f"[status] {status.state}: {text}" if text else f"[status] {status.state}")
    elif kind == "artifact-update":
        art = event.artifact
        text = None
        if art.parts:
            root = getattr(art.parts[0], "root", None)
            text = getattr(root, "text", None)
        if text:
            # Try to pretty-print JSON artifacts
            try:
                parsed = json.loads(text)
                if art.name == "tool_call_log":
                    output_summary = parsed.get("output_summary", "")
                    if isinstance(output_summary, str):
                        output_summary = output_summary.replace("\\n", "\n")
                    text = (
                        f"tool_name: {parsed.get('tool_name', '')}\n"
                        f"category: {parsed.get('category', '')}\n"
                        f"arguments: {parsed.get('arguments', '')}\n"
                        f"output_summary:\n{output_summary}\n"
                        f"extracted_tool_slugs: {parsed.get('extracted_tool_slugs', [])}"
                    )
                else:
                    text = json.dumps(parsed, indent=2)
            except Exception:
                text = text.replace("\\n", "\n")
            if len(text) > 300:
                text = text[:300] + "..."
        print(f"[artifact] {art.name}:\n{text}" if text else f"[artifact] {art.name}")


@contextlib.contextmanager
def run_a2a_server(app: A2AStarletteApplication, port: int = 8010):
    """Run A2A server in background thread."""
    config = uvicorn.Config(app.build(), host="127.0.0.1", port=port, log_level="error")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=lambda: asyncio.run(server.serve()), daemon=True)
    thread.start()
    time.sleep(2)
    print(f"✅ A2A server running on port {port}")
    try:
        yield server
    finally:
        server.should_exit = True
        time.sleep(1.5)  # Give server time to cleanup gracefully
        thread.join(timeout=6)
        if thread.is_alive():
            server.force_exit = True
            time.sleep(0.8)


async def call_a2a_agent(message: str, port: int = 8010):
    """Call A2A agent."""
    async with httpx.AsyncClient(timeout=120.0) as httpx_client:
        card_resp = await httpx_client.get(f"http://127.0.0.1:{port}{AGENT_CARD_WELL_KNOWN_PATH}")
        client = ClientFactory(
            ClientConfig(httpx_client=httpx_client, supported_transports=[TransportProtocol.jsonrpc])
        ).create(AgentCard(**card_resp.json()))

        async for evt in client.send_message(create_text_message_object(content=message)):
            pretty_event(evt)
        # Allow SSE streams to close cleanly before server shutdown
        await asyncio.sleep(0.2)

print("✅ Utility functions loaded")

✅ Utility functions loaded


## 11. Demo 1: Discovery Only Policy

Agent attempts execution but is blocked by policy.

In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    # Setup
    llm = ChatOpenAI(
        model="google/gemini-2.0-flash-001",
        openai_api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        default_headers={"HTTP-Referer": "http://localhost", "X-Title": "A2A Governed MCP"}
    )

    agent_card = AgentCard(
        name="Governed MCP Agent",
        url="http://127.0.0.1:8010",
        description="A2A Governor with policy enforcement",
        version="1.0",
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        preferred_transport=TransportProtocol.jsonrpc,
        skills=[AgentSkill(
            id="governed", name="Governed Access",
            description="Policy-enforced tool access",
            tags=["governance"], examples=["Find tools"]
        )]
    )

    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"

    app = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=DefaultRequestHandler(
            agent_executor=GovernedMCPExecutor(
                policy=DISCOVERY_ONLY_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm,
                use_llm_proposal=True
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(app, port=8010):
        print("\\n" + "="*60)
        print("Demo 1: Discovery Only Policy")
        print("Query: Normal web search is allowed")
        print("="*60)
        await call_a2a_agent("What is the current weather in New York?", port=8010)
        await asyncio.sleep(0.5)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")

✅ A2A server running on port 8010
\n============================================================
Demo 1: Discovery Only Policy
Query: Normal web search is allowed
[task] 3a3d87da submitted
[status] TaskState.working: 🛡️ Governor active: Discovery Only
[status] TaskState.working: 🔌 MCP connection will be established on demand
[status] TaskState.working: 🔧 Executing EXA_WEB_SEARCH
[artifact] tool_call_log:
tool_name: EXA_WEB_SEARCH
category: discovery
arguments: {'query': 'What is the current weather in New York?', 'num_results': 5}
output_summary:
{
  "results": [
    {
      "title": "New York, New York, United States Weather Today, Mar 21 | 51\u00b0F, Partly cloudy",
      "url": "https://exa.ai/li...
[artifact] policy_decision:
{
  "tool_name": "EXA_WEB_SEARCH",
  "allowed": true,
  "decision_reason": "Executed successfully",
  "output_summary": "{\n  \"results\": [\n    {\n      \"title\": \"New York, New York, United States Weather Today, Mar 21 | 51\\u00b0F, Partly cloudy\",\n    

## 11b. Demo: PRP skill narrows allowed arguments

Same **Discovery Only** policy as Demo 1, plus `research_strict` skill: queries whose **tool arguments** contain `stackoverflow` or `stack overflow` are denied even though `EXA_WEB_SEARCH` is policy-allowed. Normal weather-style queries still pass.


In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"

    app_prp = A2AStarletteApplication(
        agent_card=AgentCard(
            name="Governed MCP + PRP Skill",
            url="http://127.0.0.1:8015",
            description="Discovery policy + research_strict skill (argument narrowing)",
            version="1.0",
            capabilities=AgentCapabilities(streaming=True),
            default_input_modes=["text/plain"],
            default_output_modes=["text/plain"],
            preferred_transport=TransportProtocol.jsonrpc,
            skills=[AgentSkill(
                id="prp_skill", name="Research strict",
                description="Forbids stackoverflow substrings in tool args",
                tags=["governance", "prp"], examples=["stackoverflow in query → denied"]
            )]
        ),
        http_handler=DefaultRequestHandler(
            agent_executor=GovernedMCPExecutor(
                policy=DISCOVERY_ONLY_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm,
                use_llm_proposal=False,
                skill_constraints=RESEARCH_STRICT_SKILL,
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(app_prp, port=8015):
        print("\n" + "="*60)
        print("Demo 11b: PRP skill blocks stackoverflow in tool arguments")
        print("="*60)
        await call_a2a_agent(
            "Search stackoverflow for best practices on Python logging configuration",
            port=8015,
        )
        await asyncio.sleep(0.5)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")


✅ A2A server running on port 8015

Demo 11b: PRP skill blocks stackoverflow in tool arguments
[task] 6941448c submitted
[status] TaskState.working: 🛡️ Governor active: Discovery Only
[status] TaskState.working: 🔌 MCP connection will be established on demand
[artifact] policy_decision:
{
  "tool_name": "EXA_CODE_SEARCH",
  "allowed": false,
  "decision_reason": "Tool forbidden by policy",
  "output_summary": "",
  "policy_match": "Exa code search blocked",
  "policy_pattern": "EXA_CODE_SEARCH",
  "extracted_tool_slugs": [],
  "budget_stats": {
    "elapsed_seconds": 0.08,
    "tot...
[artifact] result_summary:
{
  "final_answer": "Tool blocked: EXA_CODE_SEARCH - forbidden by policy",
  "allowed_tools": [],
  "denied_tools": [
    "EXA_CODE_SEARCH"
  ],
  "allowed_outputs": [],
  "transition_summary": ""
}
[artifact] budget_stats:
{
  "elapsed_seconds": 0.09,
  "total_calls": 0,
  "active_calls": 0,
  "remaining_calls": 6,
  "remaining_runtime": 119.91,
  "tool_calls": {},
  "did_connect

## 12. Demo 2: Code Search Blocked

Agent attempts a StackOverflow/code query and is blocked by policy.

In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"

    app2 = A2AStarletteApplication(
        agent_card=AgentCard(
            name="Code Search Blocked Agent",
            url="http://127.0.0.1:8011",
            description="Blocks code search queries",
            version="1.0",
            capabilities=AgentCapabilities(streaming=True),
            default_input_modes=["text/plain"],
            default_output_modes=["text/plain"],
            preferred_transport=TransportProtocol.jsonrpc,
            skills=[AgentSkill(
                id="code_block", name="Code Search Blocked",
                description="Blocks StackOverflow/code search",
                tags=["governance"], examples=["Find how to code X"]
            )]
        ),
        http_handler=DefaultRequestHandler(
            agent_executor=GovernedMCPExecutor(
                policy=DISCOVERY_ONLY_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(app2, port=8011):
        print("\\n" + "="*60)
        print("Demo 2: Code Search Blocked")
        print("Query: How to code X on StackOverflow")
        print("="*60)
        await call_a2a_agent("Look up on StackOverflow how to code a Python decorator", port=8011)
        await asyncio.sleep(0.5)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")

✅ A2A server running on port 8011
\n============================================================
Demo 2: Code Search Blocked
Query: How to code X on StackOverflow
[task] c2738494 submitted
[status] TaskState.working: 🛡️ Governor active: Discovery Only
[status] TaskState.working: 🔌 MCP connection will be established on demand
[artifact] policy_decision:
{
  "tool_name": "EXA_CODE_SEARCH",
  "allowed": false,
  "decision_reason": "Tool forbidden by policy",
  "output_summary": "",
  "policy_match": "Exa code search blocked",
  "policy_pattern": "EXA_CODE_SEARCH",
  "extracted_tool_slugs": [],
  "budget_stats": {
    "elapsed_seconds": 0.05,
    "tot...
[artifact] result_summary:
{
  "final_answer": "Tool blocked: EXA_CODE_SEARCH - forbidden by policy",
  "allowed_tools": [],
  "denied_tools": [
    "EXA_CODE_SEARCH"
  ],
  "allowed_outputs": [],
  "transition_summary": ""
}
[artifact] budget_stats:
{
  "elapsed_seconds": 0.05,
  "total_calls": 0,
  "active_calls": 0,
  "remaining_calls":

## 13. Demo 3: Budget Exhaustion

Agent loops searching repeatedly until budget is exhausted.

In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"
    app3 = A2AStarletteApplication(
        agent_card=AgentCard(
            name="Budget Limited Agent",
            url="http://127.0.0.1:8012",
            description="Strict budget limits",
            version="1.0",
            capabilities=AgentCapabilities(streaming=True),
            default_input_modes=["text/plain"],
            default_output_modes=["text/plain"],
            preferred_transport=TransportProtocol.jsonrpc,
            skills=[AgentSkill(
                id="budget", name="Budget Policy",
                description="Limited tool calls",
                tags=["governance"], examples=["Iterate and search"]
            )]
        ),
        http_handler=DefaultRequestHandler(
            agent_executor=GovernedMCPExecutor(
                policy=BUDGET_LIMITED_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(app3, port=8012):
        print("\\n" + "="*60)
        print("Demo 3: Budget Exhaustion")
        print("Query: Iterate on web search repeatedly")
        print("="*60)
        await call_a2a_agent("Repeat web search with small refinements until confident", port=8012)
        await asyncio.sleep(0.5)

        print("\n" + "="*60)
        print("Demo 4: Separation Rule")
        print("Query: Connect then execute (should be blocked)")
        print("="*60)
        sep_app = A2AStarletteApplication(
            agent_card=AgentCard(
                name="Separation Policy Agent",
                url="http://127.0.0.1:8014",
                description="Connection/execution separation demo",
                version="1.0",
                capabilities=AgentCapabilities(streaming=True),
                default_input_modes=["text/plain"],
                default_output_modes=["text/plain"],
                preferred_transport=TransportProtocol.jsonrpc,
                skills=[AgentSkill(
                    id="separation", name="Separation Policy",
                    description="Connection then execution blocked",
                    tags=["governance"], examples=["Connect then execute"]
                )]
            ),
            http_handler=DefaultRequestHandler(
                agent_executor=GovernedMCPExecutor(
                    policy=SEPARATION_POLICY,
                    mcp_url=serpapi_url,
                    mcp_headers={},
                    llm=llm,
                    force_separation_demo=True
                ),
                task_store=InMemoryTaskStore()
            )
        )
        with run_a2a_server(sep_app, port=8014):
            await call_a2a_agent("Connect and then execute a search", port=8014)
            await asyncio.sleep(0.5)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")

✅ A2A server running on port 8012
\n============================================================
Demo 3: Budget Exhaustion
Query: Iterate on web search repeatedly
[task] b3311842 submitted
[status] TaskState.working: 🛡️ Governor active: Budget Limited
[status] TaskState.working: 🔌 MCP connection will be established on demand
[status] TaskState.working: 🔧 Executing EXA_WEB_SEARCH


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-52' coro=<_shutdown_watcher() running at /usr/local/lib/python3.12/dist-packages/sse_starlette/sse.py:107> wait_for=<Future pending cb=[Task.__wakeup()]>>
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-80' coro=<_shutdown_watcher() running at /usr/local/lib/python3.12/dist-packages/sse_starlette/sse.py:107> wait_for=<Future pending cb=[Task.__wakeup()]>>


[artifact] tool_call_log:
tool_name: EXA_WEB_SEARCH
category: discovery
arguments: {'query': 'Repeat web search with small refinements until confident', 'num_results': 5}
output_summary:
{
  "results": [
    {
      "title": "SearchReSearch: March 2026",
      "url": "https://searchresearch1.blogspot.com/2026/03/"
    },
   ...
[artifact] policy_decision:
{
  "tool_name": "EXA_WEB_SEARCH",
  "allowed": true,
  "decision_reason": "Executed successfully",
  "output_summary": "{\n  \"results\": [\n    {\n      \"title\": \"SearchReSearch: March 2026\",\n      \"url\": \"https://searchresearch1.blogspot.com/2026/03/\"\n    },\n    {\n      \"title\": \"Q...
[status] TaskState.working: 🔧 Executing EXA_WEB_SEARCH
[artifact] tool_call_log:
tool_name: EXA_WEB_SEARCH
category: discovery
arguments: {'query': 'Repeat web search with small refinements until confident (refine 1)', 'num_results': 3}
output_summary:
{
  "results": [
    {
      "title": "How to Refine Search Results for Better Accura

## 14. Demo 5: HITL Code Search Approval

Human-in-the-loop approval via LangGraph interrupt, then resume with approval.

In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"

    hitl_app = A2AStarletteApplication(
        agent_card=AgentCard(
            name="HITL Code Search Agent",
            url="http://127.0.0.1:8015",
            description="Interrupts for approval, then resumes",
            version="1.0",
            capabilities=AgentCapabilities(streaming=True),
            default_input_modes=["text/plain"],
            default_output_modes=["text/plain"],
            preferred_transport=TransportProtocol.jsonrpc,
            skills=[AgentSkill(
                id="hitl", name="HITL Code Search",
                description="Approval required for code search",
                tags=["governance"], examples=["How to code X"]
            )]
        ),
        http_handler=DefaultRequestHandler(
            agent_executor=GovernedMCPExecutor(
                policy=HITL_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm,
                use_llm_proposal=True,
                approval_on_pause=None,
                notebook_mode=True
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(hitl_app, port=8015):
        print("\n" + "="*60)
        print("Demo 5: HITL Code Search Approval")
        print("Query: Code search requires human approval")
        print("="*60)
        await call_a2a_agent("Look up on StackOverflow how to write a Python decorator", port=8015)
        await asyncio.sleep(0.5)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")

✅ A2A server running on port 8015

Demo 5: HITL Code Search Approval
Query: Code search requires human approval
[task] 69209d9a submitted
[status] TaskState.working: 🛡️ Governor active: HITL Code Search
[status] TaskState.working: 🔌 MCP connection will be established on demand
[artifact] approval_request:
{
  "tool_name": "EXA_CODE_SEARCH",
  "arguments": {
    "query": "Look up on StackOverflow how to write a Python decorator",
    "num_results": 5
  },
  "rationale": "LLM proposed ToolRequest",
  "timestamp": 1774105565.277843
}
[status] TaskState.working: ⏸️  Approval required - graph paused
Approve this tool call? (y/n): y


[artifact] approval_log:
{
  "tool_name": "EXA_CODE_SEARCH",
  "approved": true,
  "timestamp": 1774105571.0778115,
  "reason": "notebook input"
}
[status] TaskState.working: ✅ Approval granted (notebook input)
[status] TaskState.working: 🔧 Executing EXA_CODE_SEARCH
[artifact] tool_call_log:
tool_name: EXA_CODE_SEARCH
category: discovery
arguments: {'query': 'Look up on StackOverflow how to write a Python decorator', 'num_results': 5}
output_summary:
{
  "results": [
    {
      "title": "Python decorator? - can someone please explain this?",
      "url": "https://stackoverflow.com/ques...
[artifact] policy_decision:
{
  "tool_name": "EXA_CODE_SEARCH",
  "allowed": true,
  "decision_reason": "Executed successfully",
  "output_summary": "{\n  \"results\": [\n    {\n      \"title\": \"Python decorator? - can someone please explain this?\",\n      \"url\": \"https://stackoverflow.com/questions/12046883/python-decor...
[artifact] result_summary:
{
  "final_answer": "Completed tool calls. C

## 14. Summary

### What We Built

- **OmegaConf YAML packs** compose shared defaults with profile overrides; `DISCOVERY_ONLY_POLICY` loads from merged YAML under the resolved config root (`GOVERNANCE_CONFIG_DIR`, `./config` from [Google Drive](https://drive.google.com/drive/folders/17gH3w247NtejX4911wFGXQ_8dPUVlHhe?usp=sharing), or bundled `demos/config`). OmegaConf is used for **structured merge**, **env interpolation**, and **reviewable policy files**, then `to_container(resolve=True)` feeds the existing governor dataclasses.
- **PRP-style skill** (`research_strict`) adds argument-level forbiddens enforced in `governed_call` and `check_approval_node`.
- **Pydantic** validates LLM JSON via `ToolRequestModel` before building `ToolRequest`.


This notebook demonstrates **sandboxing tool execution** through governance boundaries:

1. **ToolRequest/ToolResult Schemas**: LangGraph emits structured requests, never calls MCP directly
2. **governed_call()**: Single enforcement point for all MCP access
3. **Tool Categorization**: Explicit categories (discovery, connection, execution, other) for separation rules
4. **Three Focused Policies**:
   - Discovery Only: Blocks execution tools
   - Restricted SerpApi: Approval required for SerpApi execution
   - Budget Limited: Max 3 calls for exhaustion testing
5. **Budget Tracker**: Enforces limits on calls, parallelism, runtime, and separation
6. **Argument Validation**: Gates on Exa/SerpApi queries (length limits, result limits)
7. **Approval Workflow**: Integrated with A2A events (status + artifacts)
8. **Structured Artifacts**: Only specific types (policy_decision, tool_call_log, approval_request, approval_log, budget_stats, result_summary)
9. **Audit Report Generator**: Machine-readable reports from A2A artifacts with enforcement insights

### Architecture Benefits

- **No Direct MCP Access**: Model never bypasses governor
- **On-Demand MCP Connection**: MCP session opens only when a tool call is allowed
- **Single Enforcement Point**: All decisions in governed_call()
- **Machine-Readable Audit**: Every decision logged as JSON artifact
- **Category-Based Separation**: Registry populated from discovery with safe defaults
- **Fail Closed**: Default deny for unknown tools
- **Defense in Depth**: Policy + budget + validation + approval + artifact filtering

### Threat Model

Tool calls are privileged execution. MCP increases capability density. Failures happen from normal planning:
- **Runaway loops**: Budget limits prevent infinite tool calls
- **Repeated calls**: Per-tool limits stop excessive use
- **Broad actions**: Argument gates block bulk operations
- **Capability escalation**: Policies define reachable surface area
- **Separation violations**: Connection/execution tracked and enforced

### Demo Results

1. **Discovery Only**: Normal web search ✅ (Exa web search allowed)
2. **Code Search Blocked**: StackOverflow/code query ❌ (Exa code search blocked)
3. **Budget Exhaustion**: Agent loops Exa web search ❌ (blocked after 3 calls)
4. **Separation Rule**: Connection then execution ❌ (separation violation)
5. **HITL Approval**: Code search approved via interrupt ✅
6. **Audit Report**: Comprehensive machine-readable report from all A2A artifacts

### Key Insights

- **Structured sandboxing** beats prompt engineering
- **Explicit categories** beat substring matching
- **Argument gates** prevent runaway behavior at request time
- **A2A events** provide real-time auditability
- **ToolRequest objects** prevent accidental bypass
- **governed_call()** ensures every decision is logged
- **Audit reports** enable compliance and debugging

### Audit Reporting

The `AuditReportGenerator` collects A2A artifacts and produces comprehensive reports showing:
- **Summary statistics**: Allowed vs denied requests, success rates
- **Budget usage**: Total calls, elapsed time, per-tool breakdown, separation status
- **Executed tool calls**: Detailed logs with timestamps, arguments, outputs
- **Policy decisions**: Every allow/deny decision with reasons and budget snapshots
- **Approval workflow**: All approval requests and decisions
- **Enforcement insights**: Denial breakdowns and patterns
- **Recommendations**: Actionable insights based on execution patterns

This provides **machine-readable audit trails** for compliance, debugging, and policy refinement.

### Production Extensions

- Integrate real approval systems (Slack, email, tickets)
- Add policy version control and rollback
- Implement per-user/per-tenant policies
- Create policy testing framework
- Add real-time monitoring and alerting
- Export audit reports to compliance systems (SOC2, GDPR)
- Build policy analytics dashboard
- Extend to multi-agent coordination

The notebook proves that **governance via structured enforcement** is both practical and necessary for MCP-scale tool ecosystems.


## 15. Audit Report Generator

Generate a comprehensive report from A2A artifacts showing all tool calls, decisions, and budget usage.

In [ ]:
class AuditReportGenerator:
    """Generate audit reports from A2A artifacts."""

    def __init__(self):
        self.tool_calls: List[Dict[str, Any]] = []
        self.policy_decisions: List[Dict[str, Any]] = []
        self.approval_requests: List[Dict[str, Any]] = []
        self.approval_logs: List[Dict[str, Any]] = []
        self.budget_snapshots: List[Dict[str, Any]] = []

    def add_artifact(self, artifact_name: str, artifact_content: str):
        """Add an artifact to the report."""
        try:
            data = json.loads(artifact_content)

            if artifact_name == "tool_call_log":
                self.tool_calls.append(data)
            elif artifact_name == "policy_decision":
                self.policy_decisions.append(data)
            elif artifact_name == "approval_request":
                self.approval_requests.append(data)
            elif artifact_name == "approval_log":
                self.approval_logs.append(data)
            elif artifact_name == "budget_stats":
                self.budget_snapshots.append(data)
        except json.JSONDecodeError:
            pass  # Skip non-JSON artifacts

    def generate_report(self) -> str:
        """Generate a comprehensive audit report."""
        lines = []
        lines.append("=" * 80)
        lines.append("AUDIT REPORT: Governed Tool Execution")
        lines.append("=" * 80)
        lines.append("")

        # Summary statistics
        lines.append("## Summary")
        lines.append("")
        total_decisions = len(self.policy_decisions)
        allowed = sum(1 for d in self.policy_decisions if d.get("allowed", False))
        denied = total_decisions - allowed
        lines.append(f"Total tool requests: {total_decisions}")
        lines.append(f"  ✅ Allowed: {allowed}")
        lines.append(f"  ❌ Denied: {denied}")
        lines.append(f"  📊 Success rate: {allowed/total_decisions*100:.1f}%" if total_decisions > 0 else "  📊 Success rate: N/A")
        lines.append("")

        # Approval statistics
        if self.approval_logs:
            lines.append(f"Approval requests: {len(self.approval_logs)}")
            approved = sum(1 for a in self.approval_logs if a.get("approved", False))
            lines.append(f"  ✅ Approved: {approved}")
            lines.append(f"  ❌ Denied: {len(self.approval_logs) - approved}")
            lines.append("")

        # Budget usage
        final_budget = None
        if self.budget_snapshots:
            final_budget = self.budget_snapshots[-1]
            lines.append("## Budget Usage")
            lines.append("")
            lines.append(f"Total calls made: {final_budget.get('total_calls', 0)}")
            lines.append(f"Elapsed time: {final_budget.get('elapsed_seconds', 0):.2f}s")
            lines.append(f"Connection step: {'Yes' if final_budget.get('did_connection_step') else 'No'}")
            lines.append(f"Execution step: {'Yes' if final_budget.get('did_execution_step') else 'No'}")

            tool_calls = final_budget.get('tool_calls', {})
            if tool_calls:
                lines.append("")
                lines.append("Per-tool usage:")
                for tool, count in sorted(tool_calls.items()):
                    lines.append(f"  - {tool}: {count} call(s)")
            lines.append("")

        # Detailed tool call log
        if self.tool_calls:
            lines.append("## Executed Tool Calls")
            lines.append("")
            for i, call in enumerate(self.tool_calls, 1):
                lines.append(f"### Call {i}: {call.get('tool_name', 'Unknown')}")
                lines.append(f"Category: {call.get('category', 'unknown')}")
                lines.append(f"Timestamp: {call.get('timestamp', 'N/A')}")
                lines.append(f"Arguments: {call.get('arguments', 'N/A')}")
                lines.append(f"Output: {call.get('output_summary', 'N/A')[:200]}")
                lines.append("")

        # Policy decision log
        if self.policy_decisions:
            lines.append("## Policy Decisions")
            lines.append("")
            for i, decision in enumerate(self.policy_decisions, 1):
                status = "✅ ALLOWED" if decision.get("allowed") else "❌ DENIED"
                lines.append(f"### Decision {i}: {status}")
                lines.append(f"Tool: {decision.get('tool_name', 'Unknown')}")
                lines.append(f"Reason: {decision.get('decision_reason', 'N/A')}")
                lines.append(f"Policy match: {decision.get('policy_match', 'N/A')}")

                budget = decision.get('budget_stats', {})
                if budget:
                    lines.append(f"Budget at decision: {budget.get('total_calls', 0)} calls, {budget.get('elapsed_seconds', 0):.2f}s elapsed")
                lines.append("")

        # Approval log
        if self.approval_requests or self.approval_logs:
            lines.append("## Approval Workflow")
            lines.append("")
            for i, approval in enumerate(self.approval_requests, 1):
                lines.append(f"### Approval Request {i}")
                lines.append(f"Tool: {approval.get('tool_name', 'Unknown')}")
                lines.append(f"Timestamp: {approval.get('timestamp', 'N/A')}")
                lines.append("")
            for i, approval in enumerate(self.approval_logs, 1):
                status = "✅ APPROVED" if approval.get("approved") else "❌ DENIED"
                lines.append(f"### Approval Decision {i}: {status}")
                lines.append(f"Tool: {approval.get('tool_name', 'Unknown')}")
                lines.append(f"Timestamp: {approval.get('timestamp', 'N/A')}")
                lines.append(f"Reason: {approval.get('reason', 'N/A')}")
                if "rationale" in approval:
                    lines.append(f"Rationale: {approval.get('rationale', 'N/A')}")
                lines.append("")

        # Enforcement insights
        lines.append("## Enforcement Insights")
        lines.append("")

        # Identify denial reasons
        denial_reasons = {}
        for d in self.policy_decisions:
            if not d.get("allowed", False):
                reason = d.get("decision_reason", "Unknown")
                key = reason.split(":")[0] if ":" in reason else reason
                denial_reasons[key] = denial_reasons.get(key, 0) + 1

        if denial_reasons:
            lines.append("Denial breakdown:")
            for reason, count in sorted(denial_reasons.items(), key=lambda x: -x[1]):
                lines.append(f"  - {reason}: {count} time(s)")
            lines.append("")

        # Recommendations
        lines.append("## Recommendations")
        lines.append("")

        if denied > allowed and total_decisions > 0:
            lines.append("⚠️  High denial rate detected:")
            lines.append("   - Review policy to ensure necessary tools are allowed")
            lines.append("   - Check if agent is requesting appropriate tools")

        if final_budget:
            if final_budget.get('did_connection_step') and final_budget.get('did_execution_step'):
                lines.append("⚠️  Connection and execution in same run:")
                lines.append("   - Consider enforcing separation policy")

            remaining = final_budget.get('remaining_calls', 0)
            if remaining <= 0:
                lines.append("⚠️  Budget limit reached:")
                lines.append("   - Agent may need higher limits for this task")

        if denied == 0 and self.tool_calls:
            lines.append("✅ All tool requests approved and executed successfully")

        lines.append("")
        lines.append("=" * 80)
        lines.append("End of Audit Report")
        lines.append("=" * 80)

        return "\n".join(lines)  # Use actual newline, not escaped


# Example: Simulate collecting artifacts from an A2A task
print("✅ Audit report generator implemented")
print("")
print("Example usage:")
print("```python")
print("# During task execution, collect artifacts:")
print("reporter = AuditReportGenerator()")
print("reporter.add_artifact('policy_decision', policy_json)")
print("reporter.add_artifact('tool_call_log', tool_log_json)")
print("reporter.add_artifact('budget_stats', budget_json)")
print("")
print("# Generate report:")
print("report = reporter.generate_report()")
print("print(report)")
print("```")

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-223' coro=<_shutdown_watcher() running at /usr/local/lib/python3.12/dist-packages/sse_starlette/sse.py:107> wait_for=<Future pending cb=[Task.__wakeup()]>>


✅ Audit report generator implemented

Example usage:
```python
# During task execution, collect artifacts:
reporter = AuditReportGenerator()
reporter.add_artifact('policy_decision', policy_json)
reporter.add_artifact('tool_call_log', tool_log_json)
reporter.add_artifact('budget_stats', budget_json)

# Generate report:
report = reporter.generate_report()
print(report)
```


In [ ]:
if OPENROUTER_API_KEY and EXA_API_KEY and SERPAPI_API_KEY:
    # Modified A2A executor that collects artifacts for reporting
    class AuditingGovernedMCPExecutor(GovernedMCPExecutor):
        """Governor that collects artifacts for audit reporting."""

        def __init__(self, policy, mcp_url, mcp_headers, llm, reporter: AuditReportGenerator):
            super().__init__(policy, mcp_url, mcp_headers, llm)
            self.reporter = reporter

        async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
            """Execute with artifact collection."""
            user_query = context.get_user_input().strip()
            task = context.current_task or new_task(context.message)
            await event_queue.enqueue_event(task)

            updater = TaskUpdater(event_queue, task.id, task.context_id)

            async def add_artifact(name: str, text: str) -> None:
                if name in self.policy.allowed_artifact_types:
                    await updater.add_artifact([Part(root=TextPart(text=text))], name=name)

            # Initialize components
            budget = BudgetTracker(self.policy)
            validator = ArgumentValidator()
            mcp_session = MCPSessionManager(self.mcp_url, self.mcp_headers)

            try:
                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message(f"🛡️ Governor active: {self.policy.name}", task.context_id, task.id)
                )

                await updater.update_status(
                    TaskState.working,
                    new_agent_text_message("🔌 MCP connection will be established on demand", task.context_id, task.id)
                )

                # Create LangGraph agent
                agent = create_governed_agent(
                    self.policy, budget, mcp_session,
                    validator, updater, self.llm
                )

                # Run agent with checkpointing
                self.thread_id = task.id
                config = {"configurable": {"thread_id": self.thread_id}}

                initial_state = {
                    "query": user_query,
                    "policy_summary": self.policy.name,
                    "messages": [],
                    "tool_results": [],
                    "tools_used": 0,
                    "final_answer": None,
                    "pending_request": None,
                    "needs_approval": False,
                    "approval_granted": None,
                    "discovered_tool_slugs": [],
                    "did_connection_step": False,
                    "did_execution_step": False,
                    "use_llm_proposal": False,
                    "force_separation_demo": False
                }

                final_state = await agent.ainvoke(initial_state, config)

                if "__interrupt__" in final_state:
                    interrupt_payload = final_state.get("__interrupt__", [])[0].value
                    approval_req_json = json.dumps(interrupt_payload, indent=2)
                    await add_artifact("approval_request", approval_req_json)
                    self.reporter.add_artifact("approval_request", approval_req_json)
                    await updater.update_status(
                        TaskState.working,
                        new_agent_text_message("⏸️  Approval required - graph paused", task.context_id, task.id)
                    )
                    budget.pause_timer()
                    approved = False
                    approval_log = {
                        "tool_name": interrupt_payload.get("tool_name"),
                        "approved": approved,
                        "timestamp": time.time(),
                        "reason": "demo denial"
                    }
                    approval_log_json = json.dumps(approval_log, indent=2)
                    await add_artifact("approval_log", approval_log_json)
                    self.reporter.add_artifact("approval_log", approval_log_json)
                    await updater.update_status(
                        TaskState.working,
                        new_agent_text_message("❌ Approval denied (demo)", task.context_id, task.id)
                    )
                    budget.resume_timer()
                    final_state = await agent.ainvoke(Command(resume=approved), config)

                # Collect budget stats for report
                budget_json = json.dumps(budget.get_stats(), indent=2)
                self.reporter.add_artifact("budget_stats", budget_json)

                # Collect tool results for report
                for tr in final_state.get("tool_results", []):
                    self.reporter.add_artifact("policy_decision", tr.to_json())

                # Emit final result (clean allowed vs denied)
                tool_results = final_state.get("tool_results", [])
                allowed_tools = [tr.tool_name for tr in tool_results if tr.allowed]
                denied_tools = [tr.tool_name for tr in tool_results if not tr.allowed]
                allowed_outputs = []
                for tr in tool_results:
                    if not tr.allowed or not tr.output_summary:
                        continue
                    summary_text = tr.output_summary[:200]
                    try:
                        parsed = json.loads(tr.output_summary)
                        results = parsed.get("results", []) if isinstance(parsed, dict) else []
                        if results:
                            summary_text = "; ".join(
                                f"{r.get('title', '')} ({r.get('url', '')})"
                                for r in results
                                if isinstance(r, dict)
                            )[:200]
                    except Exception:
                        pass
                    allowed_outputs.append({"tool": tr.tool_name, "summary": summary_text})

                transition_summary = ""
                if "EXA_CODE_SEARCH" in denied_tools and "EXA_WEB_SEARCH" in allowed_tools:
                    transition_summary = "code search blocked → web search allowed"

                summary_payload = {
                    "final_answer": final_state.get("final_answer", "No answer"),
                    "allowed_tools": allowed_tools,
                    "denied_tools": denied_tools,
                    "allowed_outputs": allowed_outputs,
                    "transition_summary": transition_summary
                }
                await add_artifact("result_summary", json.dumps(summary_payload, indent=2))

                await add_artifact("budget_stats", budget_json)

                await updater.complete()

            except Exception as e:
                await updater.update_status(
                    TaskState.failed,
                    new_agent_text_message(f"❌ Error: {str(e)}", task.context_id, task.id),
                    final=True
                )
            finally:
                pass  # MCP disconnect handled in graph cleanup node

    # Create reporter and run demo
    reporter = AuditReportGenerator()

    serpapi_url = f"https://mcp.serpapi.com/{SERPAPI_API_KEY}/mcp"

    app_audit = A2AStarletteApplication(
        agent_card=AgentCard(
            name="Audited Governed Agent",
            url="http://127.0.0.1:8013",
            description="Governed agent with audit reporting",
            version="1.0",
            capabilities=AgentCapabilities(streaming=True),
            default_input_modes=["text/plain"],
            default_output_modes=["text/plain"],
            preferred_transport=TransportProtocol.jsonrpc,
            skills=[AgentSkill(
                id="audited", name="Audited Execution",
                description="Tracked and reported execution",
                tags=["governance", "audit"], examples=["Find tools"]
            )]
        ),
        http_handler=DefaultRequestHandler(
            agent_executor=AuditingGovernedMCPExecutor(
                policy=DISCOVERY_ONLY_POLICY,
                mcp_url=serpapi_url,
                mcp_headers={},
                llm=llm,
                reporter=reporter
            ),
            task_store=InMemoryTaskStore()
        )
    )

    with run_a2a_server(app_audit, port=8013):
        print("\\n" + "="*80)
        print("Demo: Audited Execution with Report Generation")
        print("="*80)
        await call_a2a_agent("List SerpApi tools", port=8013)
        await asyncio.sleep(0.5)

        # Generate and display report
        print("\\n" + "="*80)
        print("GENERATING AUDIT REPORT...")
        print("="*80)
        print("")

        report = reporter.generate_report()
        print(report)
else:
    print("⚠️  Set OPENROUTER_API_KEY, EXA_API_KEY, and SERPAPI_API_KEY")

✅ A2A server running on port 8013
\n================================================================================
Demo: Audited Execution with Report Generation
[task] e38b3dd5 submitted
[status] TaskState.working: 🛡️ Governor active: Discovery Only
[status] TaskState.working: 🔌 MCP connection will be established on demand
[status] TaskState.working: 🔧 Executing EXA_WEB_SEARCH
[artifact] tool_call_log:
tool_name: EXA_WEB_SEARCH
category: discovery
arguments: {'query': 'List SerpApi tools', 'num_results': 5}
output_summary:
{
  "results": [
    {
      "title": "20+ SERP APIs to Try Today [Updated List] - Lupage Digital",
      "url": "https://www.lupagedigital.com/blog/serp-api/"
    },
    {
    ...
[artifact] policy_decision:
{
  "tool_name": "EXA_WEB_SEARCH",
  "allowed": true,
  "decision_reason": "Executed successfully",
  "output_summary": "{\n  \"results\": [\n    {\n      \"title\": \"20+ SERP APIs to Try Today [Updated List] - Lupage Digital\",\n      \"url\": \"https://www